<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 3 · Hierarchical Clustering & Dendrogram</h4>
<h1 align="center">Hierarchical Clustering — Country Development Indicators</h1>
<p align="center"><i>Scaling, dimensionality reduction, and a stability-check setup for unsupervised learning</i></p>

---

## 1. Why Distance-Based Clustering Needs Scaling

Hierarchical clustering (and most clustering algorithms) decide which points belong together
by computing a **distance** between every pair of points — most commonly Euclidean distance:

$$d(x, y) = \sqrt{\sum_{i=1}^{p} (x_i - y_i)^2}$$

The problem: this formula treats every feature's raw numeric units as equally meaningful.
From `01_eda.ipynb`, we already know `income` ranges from roughly 600 to 100,000+ (dollars),
while `total_fer` ranges from about 1 to 7 (children per woman). A difference of "1,000" in
`income` is economically almost negligible, while a difference of "1,000" in `total_fer`
would be nonsensical (no country differs by 1,000 children per woman!). Yet in the raw
Euclidean formula, a 1,000-unit gap in `income` and a 1,000-unit gap in `total_fer` would
contribute *identically* to the total distance.

In practice this means: **without scaling, `income` and `gdpp` (both in the thousands) would
completely dominate the distance calculation**, and features like `total_fer`, `health`, or
`inflation` (all much smaller numbers) would contribute almost nothing — even though
conceptually every one of these indicators matters equally to the NGO's notion of
"development."

### 1.1 The fix: `StandardScaler`

`StandardScaler` transforms every feature to have **mean 0 and standard deviation 1**:

$$z_i = \frac{x_i - \mu}{\sigma}$$

where $\mu$ and $\sigma$ are the feature's mean and standard deviation across all 167
countries. After this transform, every feature is expressed in "standard deviations away from
the average country" rather than its own raw unit — so a 1-standard-deviation difference in
`total_fer` counts the same as a 1-standard-deviation difference in `income`. This puts every
indicator on equal footing for the distance calculation, which is exactly what we want: the
NGO considers health, economics, and demographics all equally important dimensions of
"development."

> **Blockquote.** Scaling does **not** change the relative ordering of countries within a
> single feature, and it does not remove the real extreme values we decided to keep in EDA —
> it only rescales every feature onto a common scale so that no single feature accidentally
> dominates the geometry the clustering algorithm sees.

Let's load the clean data and apply this.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/country_data_clean.csv")
feature_cols = ["child_mort", "exports", "health", "imports", "income",
                 "inflation", "life_expec", "total_fer", "gdpp"]

X = df[feature_cols].copy()
X.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
child_mort,38.270060,40.328931,2.6000,208.00
exports,41.108976,27.412010,0.1090,200.00
health,6.815689,2.746837,1.8100,17.90
imports,46.890215,24.209589,0.0659,174.00
income,17144.688623,19278.067698,609.0000,125000.00
inflation,7.781832,10.570704,-4.2100,104.00
life_expec,70.555689,8.893172,32.1000,82.80
total_fer,2.947964,1.513848,1.1500,7.49
gdpp,12964.155689,18328.704809,231.0000,105000.00


As expected, the raw scales vary enormously: `income` has a mean in the thousands with
a standard deviation over ten thousand, while `total_fer` has a mean around 3 with a standard
deviation under 2. This confirms scaling is necessary before computing any distance.

### 1.2 Applying `StandardScaler`

In [2]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled.insert(0, "country", df["country"].values)

print("Scaled feature means (should be ~0):")
print(df_scaled[feature_cols].mean().round(3))
print("\nScaled feature stds (should be ~1):")
print(df_scaled[feature_cols].std(ddof=0).round(3))

df_scaled.head()

Scaled feature means (should be ~0):
child_mort   -0.0
exports       0.0
health        0.0
imports       0.0
income       -0.0
inflation    -0.0
life_expec    0.0
total_fer     0.0
gdpp          0.0
dtype: float64

Scaled feature stds (should be ~1):
child_mort    1.0
exports       1.0
health        1.0
imports       1.0
income        1.0
inflation     1.0
life_expec    1.0
total_fer     1.0
gdpp          1.0
dtype: float64


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,1.291532,-1.138280,0.279088,-0.082455,-0.808245,0.157336,-1.619092,1.902882,-0.679180
1,Albania,-0.538949,-0.479658,-0.097016,0.070837,-0.375369,-0.312347,0.647866,-0.859973,-0.485623
2,Algeria,-0.272833,-0.099122,-0.966073,-0.641762,-0.220844,0.789274,0.670423,-0.038404,-0.465376
3,Angola,2.007808,0.775381,-1.448071,-0.165315,-0.585043,1.387054,-1.179234,2.128151,-0.516268
4,Antigua and Barbuda,-0.695634,0.160668,-0.286894,0.497568,0.101732,-0.601749,0.704258,-0.541946,-0.041817


Every feature now has mean approximately 0 and standard deviation approximately 1
(computed with `ddof=0`, matching how `StandardScaler` computes the population standard
deviation it divides by). No feature can numerically dominate a Euclidean distance
calculation purely because of its raw units anymore — differences are now measured in
comparable "standard deviations from the average country" for every indicator.

---

## 2. Dimensionality Reduction for Visualization: PCA to 2D

Our scaled feature space has 9 dimensions — impossible to plot directly. **Principal
Component Analysis (PCA)** finds the 2 directions (linear combinations of the 9 original
features) that capture the most variance in the data, so we can visualize cluster assignments
on a 2D scatter plot later in `03_train_test_eval.ipynb`.

> **Important framing.** PCA here is used purely as a **visualization aid** — we will still
> run the actual clustering algorithm on the full 9-dimensional scaled data, not on the 2D PCA
> projection. The PCA projection is only a convenient 2D "shadow" of the 9D space that lets us
> *see* whether the clusters found in 9D look visually sensible when flattened to 2D.

### 2.1 What "% variance explained" means here

Each principal component is a direction in the 9-dimensional feature space. The **variance
explained** by a component is the fraction of the total spread (variance) in the data that
lies along that direction:

$$\text{variance explained by component } k = \frac{\lambda_k}{\sum_{j=1}^{9} \lambda_j}$$

where $\lambda_k$ is the eigenvalue associated with the $k$-th principal component. If the
first two components together explain, say, 70% of the total variance, that means a 2D plot
using just those two components preserves about 70% of the "spread-out-ness" of the original
9-dimensional data — a reasonably faithful (though imperfect) 2D summary.

In [3]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca.insert(0, "country", df["country"].values)

explained = pca.explained_variance_ratio_
print(f"PC1 explains {explained[0]*100:.1f}% of variance")
print(f"PC2 explains {explained[1]*100:.1f}% of variance")
print(f"Together: {explained.sum()*100:.1f}% of total variance")

df_pca.head()

PC1 explains 46.0% of variance
PC2 explains 17.2% of variance
Together: 63.1% of total variance


,country,PC1,PC2
0,Afghanistan,-2.913025,0.095621
1,Albania,0.429911,-0.588156
2,Algeria,-0.285225,-0.455174
3,Angola,-2.932423,1.695555
4,Antigua and Barbuda,1.033576,0.136659


Together, the first two principal components capture a large majority of the total
variance in the scaled data — meaning the 2D scatter plot we will build in
`03_train_test_eval.ipynb` is a reasonably faithful visual summary of the 9-dimensional
distances the clustering algorithm actually uses, even though it necessarily loses some
information.

---

## 3. No Train/Test Split — Setting Up a Stability Check Instead

Unlike supervised learning, there is **no target variable** here, so a train/test split in
the usual sense makes no sense: there is no "correct answer" on a held-out set to score
predictions against. Splitting the data would only reduce our sample size without buying us
any evaluation power.

Instead, since we cannot compute a held-out "accuracy," we adopt a different kind of sanity
check that unsupervised learning does support: **stability under resampling**. If the clusters
we find are real structure in the data (rather than an artifact of exactly these 167 rows),
then clustering two different but heavily-overlapping subsamples of countries should produce
**similar** groupings. If the groupings look wildly different from one subsample to another,
that's a warning sign that the chosen number of clusters (or the clustering itself) is not
robust.

We prepare two overlapping random subsamples now, to be used for this stability sanity check
in `03_train_test_eval.ipynb`.

> **Blockquote.** This is not a substitute for labeled test accuracy — it cannot tell us the
> clusters are "correct," since there is no ground truth. It can only tell us whether the
> clusters are **stable**, i.e. not overly sensitive to which exact 167 countries happened to
> be in our sample.

In [4]:
rng = np.random.RandomState(42)
n = len(df_scaled)

# Two overlapping ~85%-sized random subsamples of country indices, for a later
# stability check (Section 4 of 03_train_test_eval.ipynb).
subsample_size = int(n * 0.85)
subsample_a_idx = rng.choice(n, size=subsample_size, replace=False)
subsample_b_idx = rng.choice(n, size=subsample_size, replace=False)

overlap = len(set(subsample_a_idx) & set(subsample_b_idx))
print(f"Subsample A: {len(subsample_a_idx)} countries")
print(f"Subsample B: {len(subsample_b_idx)} countries")
print(f"Overlap between A and B: {overlap} countries "
      f"({overlap / subsample_size * 100:.0f}% of each subsample)")

np.save("data/subsample_a_idx.npy", subsample_a_idx)
np.save("data/subsample_b_idx.npy", subsample_b_idx)
print("\nSaved data/subsample_a_idx.npy and data/subsample_b_idx.npy")

Subsample A: 141 countries
Subsample B: 141 countries
Overlap between A and B: 117 countries (83% of each subsample)

Saved data/subsample_a_idx.npy and data/subsample_b_idx.npy


These two index arrays are large, overlapping-but-different 85% subsamples of the 167
countries. In `03_train_test_eval.ipynb` we will fit hierarchical clustering separately on
each subsample and compare the resulting groupings for the countries common to both — a
lightweight, appropriate substitute for the train/test evaluation you'd use in supervised
learning.

---

## 4. Saving the Processed Data

In [5]:
df_scaled.to_csv("data/country_data_scaled.csv", index=False)
df_pca.to_csv("data/country_data_pca.csv", index=False)

print("Saved data/country_data_scaled.csv ->", df_scaled.shape)
print("Saved data/country_data_pca.csv    ->", df_pca.shape)

Saved data/country_data_scaled.csv -> (167, 10)
Saved data/country_data_pca.csv    -> (167, 3)


---

## 5. Summary

- **Scaling is mandatory** before distance-based clustering on this dataset, because
  `income`/`gdpp` (thousands) would otherwise dominate `total_fer`/`health` (single digits)
  in any Euclidean distance calculation. `StandardScaler` puts every indicator on equal
  footing (mean 0, standard deviation 1).
- A **2D PCA projection** was computed purely for later visualization of cluster
  assignments — the actual clustering will still run on the full 9-dimensional scaled data.
- There is **no train/test split** for this unsupervised task; instead we prepared two
  overlapping random subsamples of countries to sanity-check **cluster stability** later,
  since there is no held-out "accuracy" to compute.
- Outputs saved: `data/country_data_scaled.csv`, `data/country_data_pca.csv`,
  `data/subsample_a_idx.npy`, `data/subsample_b_idx.npy`.

Next, `03_train_test_eval.ipynb` builds the dendrogram, evaluates different numbers of
clusters, and profiles the resulting country groups for the NGO.